In [ ]:
# (setup cell already installs what this notebook needs)

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## Guardrails

- A guardrail is a check placed around the model, outside the prompt
- Validate what goes in, and constrain what comes back out
- Cheap checks catch most of it : valid JSON, a regex, a length cap

Three of them : JSON validation, a regex match, and a word limit.

### Exercise JSON validation:
1. Run the code below and see why the supplied JSON is not valid
2. Modify the JSON so that it's valid and run the code again

In [ ]:
from langchain_classic.evaluation import JsonValidityEvaluator

evaluator = JsonValidityEvaluator()
print(evaluator.evaluate_strings(prediction='{x: 1}'))  # <- after the first run, modify the JSON here and run again

### Exercise
Regular expression \
Replace "____" with a valid order number matching the regular expression. Correct text will return an evaluation result of 1.

In [ ]:
from langchain_classic.evaluation import RegexMatchStringEvaluator

evaluator = RegexMatchStringEvaluator()
result = evaluator.evaluate_strings(
    prediction="_____", # <- insert the order number here
    reference=r"^Order ID: [A-Z]{3}-\d{4}$",
)
print(result['score'])

### Exercise
Word limit
Run the code. Then decrease the values of the variables limit and lower_limit by 5 and run the code again.
See how the model's response will differ.

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llm = make_llm()

limit = 25
prompt = f"Explain in MAX {limit} words why this refund was refused:" # <- replace the text
lower_limit = 10
resp = llm.invoke(prompt).content
if len(resp.split()) > lower_limit:
    print("The response text is too long - a summary is required. ")
    # quick fix - ask the model to shorten to the limit
    resp = llm.invoke(f"Shorten this to max {lower_limit} words, with no extra notes:\n\n{resp}").content

print(resp)

### Try tightening the limit

- Drop the word limit to something the model cannot meet and re-run
- It will overrun. A limit in the prompt is a request the model may ignore ; the
  check afterwards is what actually enforces it